# Call Qodec Instructions from OpenQASM 3

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do you invoke `qodec` instructions from an OpenQASM 3 program and assemble multi-document `.qodec` YAML bundles?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Declare `@qdk.qir.intrinsic gate` and extern measurement intrinsics in OpenQASM 3 to target a `qodec` protocol.
- Understand when custom `qodec` mnemonics are used directly versus when standard OpenQASM gate names are mapped in the top layer.
- Load and audit multi-document `.qodec` YAML bundles containing OpenQASM 3 gadget circuits.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## How OpenQASM 3 Statements Map to Qodec Instructions

With `run_qir(..., qodec=protocol)`, each quantum call in the compiled QIR invokes the instruction in the `qodec`'s top-layer instruction set whose `mnemonic` **exactly** matches the QIR callee name.

> **Important Note on Top-Layer Instruction Names (`@qdk.qir.intrinsic` vs. Standard Gates):**
> Standard single-qubit/two-qubit OpenQASM gates (`x q;`, `cx a, b;`) are **not required** in the top layer of a `qodec`—in fact, a block-encoded protocol like our $C_4$ `qodec` exposes multi-qubit block operations (`prepare_z_all`, `cx_all`, `syndrome`, `measure_z_all`) rather than standard single-qubit gate names.
> - **Calling block-encoded `qodec` instructions (`prepare_z_all`, `syndrome`, `cx_all`):** Declare them in OpenQASM 3 with `@qdk.qir.intrinsic` on the line immediately before `gate <mnemonic> ... {}` (OpenQASM 3 annotations are line-delimited), which compiles directly to a call with that exact mnemonic.
> - **Calling standard OpenQASM statements (`x q;`, `c = measure q;`):** Only when a `qodec`'s top layer intentionally defines an instruction whose mnemonic is a QIR standard intrinsic name (such as `__quantum__qis__x__body` in the unencoded tutorial `qodec_instructions_from_openqasm.ipynb`, or `__quantum__qis__m__body` when using OpenQASM's built-in `measure` statement) does that statement resolve against the `qodec`.

| OpenQASM 3 Statement | Top-Layer Qodec Mnemonic | When Used |
| :--- | :--- | :--- |
| `@qdk.qir.intrinsic gate foo q {}` | `foo` | **Primary pattern for encoded protocols:** calls custom top-layer block instructions (`prepare_z_all`, `syndrome`, `cx_all`). |
| `@qdk.qir.intrinsic def turn(float theta, qubit q) {}` | `turn` (`parameters: {theta: number}`) | Custom parameterized top-layer instruction; classical arguments bind `parameters` in order. |
| `c = measure q;` | `__quantum__qis__m__body` (`action: [observe: Z_0]`) | Required when measuring from OpenQASM 3, because OpenQASM 3 does not support `@qdk.qir.intrinsic` functions returning measurement bits (`-> bit`). |
| Standard gate (`x q;`, `cx a, b;`, `sdg q;`) | `__quantum__qis__x__body`, `__quantum__qis__cx__body`, `__quantum__qis__s__adj` | Optional: only available if the top layer of the `qodec` explicitly declares these QIR intrinsic mnemonics. |
| Instruction with `flags: [reject]` | Raised flag rejects the shot | OpenQASM calls cannot receive flags as return values, so `run_qir` automatically rejects shots where a flag is raised according to `on_shot_failure`. |

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Compile & Run an OpenQASM 3 Logical Program Against Our `C4` Protocol
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_all")
from qdk import TargetProfile, openqasm
from qdk.simulation import run_qir
from qdk.simulation.decoders import prepare_syndrome_decoder

# In OpenQASM 3, `c = measure block;` calls `__quantum__qis__m__body` (1-bit Z_0 measurement),
# while `@qdk.qir.intrinsic gate prepare_z_all q {}` calls `prepare_z_all` (with automatic flag checking!).
qasm_protocol = qodec.Qodec.loads(protocol.dumps())
qasm_layer0 = qasm_protocol.layers[0]

m_instr = qodec.Instruction(
    mnemonic="__quantum__qis__m__body",
    description="Measure logical Z_0 on C4 block.",
    inputs=[qodec.instructions.BlockOperand("C4")],
    outputs=[],
    action=[qodec.actions.Observe(["Z_0"])],
)
qasm_layer0.instruction_set.instructions["__quantum__qis__m__body"] = m_instr
qasm_layer0.gadgets["__quantum__qis__m__body"] = ec.filled(
    qodec.Gadget(
        implements=m_instr,
        inputs=list(qasm_layer0.gadgets["measure_z_all"].inputs),
        outputs=[],
        circuit=qodec.gadgets.Circuit(
            format="stim",
            source="M 0 1 2 3\n",
            instruction_set=qasm_protocol.layers[1].instruction_set,
        ),
    )
)

qasm_logical_program = """
OPENQASM 3.0;
include "stdgates.inc";

@qdk.qir.intrinsic
gate prepare_z_all q {}

@qdk.qir.intrinsic
gate syndrome q {}

qubit block;
bit c;
prepare_z_all block;
syndrome block;
c = measure block;
"""

qasm_qir = str(openqasm.compile(qasm_logical_program, target_profile=TargetProfile.Adaptive_RI))
qasm_shots = run_qir(
    qasm_qir,
    qodec=qasm_protocol,
    decoder=prepare_syndrome_decoder,
    shots=10,
    seed=11,
    type="clifford",
    on_shot_failure="discard",
)
print("OpenQASM 3 logical program compiled to QIR and executed on C4 protocol!")
print("First 5 shots (logical Z_0 readout):", qasm_shots[:5])

OpenQASM 3 logical program compiled to QIR and executed on C4 protocol!
First 5 shots (logical Z_0 readout): [Zero, Zero, Zero, Zero, Zero]


## Authoring a Custom Multi-Document `Qodec` Bundle for OpenQASM 3 Intrinsics & Checked Preparations

Now let's see the pattern from `qodec_instructions_from_openqasm.ipynb` for assembling a custom two-layer `qodec.Qodec` YAML bundle from scratch—combining:
- A verified preparation (`prepare_checked`) with `flags: [reject]` implemented by a 2-qubit CNOT check (`circuit: {source: [R: [0], R: [1], CX: [0, 1], M: [1]]}`),
- A custom `@qdk.qir.intrinsic gate flip q {}` (`action: [pauli: X_0]`), and
- Standard OpenQASM 3 `c = measure q;` (`__quantum__qis__m__body`).

In [3]:
# Multi-Document `Qodec` Bundle for OpenQASM 3 Custom Gates & Checked Prep
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_all")
from qdk import TargetProfile, openqasm
from qdk.simulation import run_qir

custom_qasm_qodec = qodec.Qodec.loads("""
---
qodec.yaml:
  name: openqasm_tutorial
  layers:
  - instruction_set: top.isa.yaml
    codes: {qubit: qubit.code.yaml}
    gadgets:
      prepare_checked: prepare_checked.gadget.yaml
      flip: flip.gadget.yaml
      __quantum__qis__m__body: m.gadget.yaml
  - instruction_set: physical.isa.yaml
---
top.isa.yaml:
  name: top
  blocks: {qubit: 1}
  instructions:
  - mnemonic: prepare_checked
    description: Prepare |0> and verify with an ancilla flag.
    out: [qubit]
    action: [stabilize: Z_0]
    flags: [reject]
  - mnemonic: flip
    description: Pauli X under a custom OpenQASM gate name.
    in: [qubit]
    out: [qubit]
    action: [pauli: X_0]
  - mnemonic: __quantum__qis__m__body
    description: Measure Z.
    in: [qubit]
    action: [observe: Z_0]
---
physical.isa.yaml:
  name: physical
  blocks: {qubit: 1}
  instructions:
  - mnemonic: R
    description: Reset to |0>.
    out: [qubit]
    action: [stabilize: Z_0]
  - mnemonic: X
    description: Pauli X.
    in: [qubit]
    out: [qubit]
    action: [pauli: X_0]
  - mnemonic: CX
    description: CNOT.
    in: [qubit, qubit]
    out: [qubit, qubit]
    action: [clifford: {X_0: X_0 X_1, Z_1: Z_0 Z_1}]
  - mnemonic: M
    description: Measure Z.
    in: [qubit]
    action: [observe: Z_0]
---
qubit.code.yaml:
  name: qubit
  description: One logical qubit.
  stabilizers: []
  x: [X_0]
  z: [Z_0]
---
prepare_checked.gadget.yaml:
  circuit: {source: [R: [0], R: [1], CX: [0, 1], M: [1]]}
  readouts: [reject: ["circuit.readouts[0]"]]
---
flip.gadget.yaml:
  circuit: {source: [X: [0]]}
---
m.gadget.yaml:
  circuit: {source: [M: [0]]}
  readouts: [["circuit.readouts[0]", "in[0].z[0]"]]
""")

print("Audit of custom OpenQASM `Qodec` bundle:")
print(ec.audit(custom_qasm_qodec))

caller_qasm = """
OPENQASM 3.0;
include "stdgates.inc";

@qdk.qir.intrinsic
gate prepare_checked q {}

@qdk.qir.intrinsic
gate flip q {}

qubit q;
bit c;
prepare_checked q;
flip q;
c = measure q;
"""
caller_qir = str(openqasm.compile(caller_qasm, target_profile=TargetProfile.Adaptive_RI))
out = run_qir(caller_qir, qodec=custom_qasm_qodec, shots=5, seed=99)
print("Result of `prepare_checked -> flip -> measure` (5 shots):", out)

Audit of custom OpenQASM `Qodec` bundle:
audit: ok


Result of `prepare_checked -> flip -> measure` (5 shots): [One, One, One, One, One]


## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why can't you declare <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">@qdk.qir.intrinsic def measure_z_all(qubit q) -&gt; bit[2] {}</code> in OpenQASM 3, and how do OpenQASM 3 and Q# differ when calling multi-outcome or flagged <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">qodec</code> instructions?</summary>

<div style="padding:0.1em 1em;">

OpenQASM 3 does not support custom-measurement intrinsics (`@qdk.qir.intrinsic def` cannot return a QIR `Result`). From OpenQASM 3, measurements use `c = measure q;` (which calls `__quantum__qis__m__body`), and raised `flags` on `@qdk.qir.intrinsic gate` calls automatically reject the shot via `on_shot_failure`. Multi-outcome block measurements (`(Result, Result)`) and explicit flag returns to the program require Q#'s `@Measurement()` attribute.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How do you declare a custom unitary or preparation <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">qodec</code> instruction inside an OpenQASM 3 program so that <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">openqasm.compile</code> emits a direct QIR call to that instruction's mnemonic?</summary>

<div style="padding:0.1em 1em;">

Annotate a `gate` declaration with `@qdk.qir.intrinsic` and an empty body `{}`, e.g., `@qdk.qir.intrinsic gate syndrome q {}`.

</div>

</details></div>

## Exercise: Execute a Two-Block OpenQASM 3 Program on C4

Compile a two-block OpenQASM 3 program that declares `@qdk.qir.intrinsic gate cx_all a, b {}`, prepares two $C_4$ blocks in $|00\rangle_L$, applies `cx_all b0, b1;`, extracts syndromes, and measures both blocks, then return the number of noiseless shots (`10`).

In [ ]:
from qdk import TargetProfile, ec, openqasm
from qdk.simulation import run_qir
from qdk.simulation.decoders import prepare_syndrome_decoder
import qodec
from _unit import exercise, simulation


@exercise
def openqasm_two_block_shots() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_all')
    qasm_proto = qodec.Qodec.loads(ns['protocol'].dumps())
    layer0 = qasm_proto.layers[0]
    m_instr = qodec.Instruction(
        mnemonic='__quantum__qis__m__body',
        inputs=[qodec.instructions.BlockOperand('C4')],
        outputs=[],
        action=[qodec.actions.Observe(['Z_0'])],
    )
    layer0.instruction_set.instructions['__quantum__qis__m__body'] = m_instr
    layer0.gadgets['__quantum__qis__m__body'] = ec.filled(
        qodec.Gadget(
            implements=m_instr,
            inputs=list(layer0.gadgets['measure_z_all'].inputs),
            outputs=[],
            circuit=qodec.gadgets.Circuit(
                format='stim',
                source='M 0 1 2 3\n',
                instruction_set=qasm_proto.layers[1].instruction_set,
            ),
        )
    )
    qasm_src = '''
    OPENQASM 3.0;
    include "stdgates.inc";
    @qdk.qir.intrinsic
    gate prepare_z_all q {}
    @qdk.qir.intrinsic
    gate cx_all a, b {}
    @qdk.qir.intrinsic
    gate syndrome q {}
    qubit b0; qubit b1; bit c0; bit c1;
    prepare_z_all b0; prepare_z_all b1;
    cx_all b0, b1;
    syndrome b0; syndrome b1;
    c0 = measure b0; c1 = measure b1;
    '''
    qir = str(openqasm.compile(qasm_src, target_profile=TargetProfile.Adaptive_RI))
    # Execute 10 noiseless shots with run_qir and return len(shots_out)
    return ...

**Hint**

Call `shots_out = run_qir(qir, qodec=qasm_proto, decoder=prepare_syndrome_decoder, shots=10, seed=11, type='clifford')` and return `len(shots_out)`.

In [ ]:
from qdk import TargetProfile, ec, openqasm
from qdk.simulation import run_qir
from qdk.simulation.decoders import prepare_syndrome_decoder
import qodec
from _unit import exercise, simulation


@exercise
def openqasm_two_block_shots() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_all')
    qasm_proto = qodec.Qodec.loads(ns['protocol'].dumps())
    layer0 = qasm_proto.layers[0]
    m_instr = qodec.Instruction(
        mnemonic='__quantum__qis__m__body',
        inputs=[qodec.instructions.BlockOperand('C4')],
        outputs=[],
        action=[qodec.actions.Observe(['Z_0'])],
    )
    layer0.instruction_set.instructions['__quantum__qis__m__body'] = m_instr
    layer0.gadgets['__quantum__qis__m__body'] = ec.filled(
        qodec.Gadget(
            implements=m_instr,
            inputs=list(layer0.gadgets['measure_z_all'].inputs),
            outputs=[],
            circuit=qodec.gadgets.Circuit(
                format='stim',
                source='M 0 1 2 3\n',
                instruction_set=qasm_proto.layers[1].instruction_set,
            ),
        )
    )
    qasm_src = '''
    OPENQASM 3.0;
    include "stdgates.inc";
    @qdk.qir.intrinsic
    gate prepare_z_all q {}
    @qdk.qir.intrinsic
    gate cx_all a, b {}
    @qdk.qir.intrinsic
    gate syndrome q {}
    qubit b0; qubit b1; bit c0; bit c1;
    prepare_z_all b0; prepare_z_all b1;
    cx_all b0, b1;
    syndrome b0; syndrome b1;
    c0 = measure b0; c1 = measure b1;
    '''
    qir = str(openqasm.compile(qasm_src, target_profile=TargetProfile.Adaptive_RI))
    shots_out = run_qir(qir, qodec=qasm_proto, decoder=prepare_syndrome_decoder, shots=10, seed=11, type='clifford')
    return len(shots_out)

Declaring `@qdk.qir.intrinsic gate cx_all a, b {}` in OpenQASM 3 emits a two-operand QIR call to `cx_all`, which `run_qir` lowers into the transversal 4-CNOT `cx_all` gadget between the two encoded $C_4$ blocks.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 8
- **Learning Objectives:**
  1. Understand the exact OpenQASM 3 to `qodec` instruction mapping rules from `qodec_instructions_from_openqasm.ipynb` (including why OpenQASM 3 uses `@qdk.qir.intrinsic gate` for preparations/unitaries and `c = measure q;` -> `__quantum__qis__m__body` for readout, while `run_qir` automatically handles `flags: [reject]` via `on_shot_failure`).
  2. Add a `__quantum__qis__m__body` instruction and gadget to a copy of our $C_4$ protocol and execute an OpenQASM 3 logical program (`prepare_z_all` $\to$ `syndrome` $\to$ `measure`) against $C_4$.
  3. Build and audit a multi-document `qodec.Qodec.loads(...)` bundle implementing custom OpenQASM 3 intrinsics (`flip`, `prepare_checked` with `flags: [reject]`, and `__quantum__qis__s__adj`).

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../07-qsharp-compilation/qsharp_compilation.workbook.ipynb" title="Previous unit: Compile Logical Experiments from Q#" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Compile Logical Experiments from Q#</span></span></a><a href="../09-simulation-and-decoding/simulation_and_decoding.workbook.ipynb" title="Next unit: Noisy Simulation, Decoding & Confidence Filtering" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Noisy Simulation, Decoding & Confidence Filtering</span></span><span aria-hidden="true">&#8250;</span></a></div>